# Step 10d-1: kiểm định ghép cặp + Holm trên cả 8 bộ, hai LLM chấm

Chạy lại đúng 4 họ kiểm định của Step 10-B2 (F1 QuDAR-llm so với Equal; F2 Qwen so với gpt-4o-mini; F3 ablation thứ tự × prompt; F4 thiên lệch vị trí), nhưng mỗi họ giờ gồm **8 bộ**: FiQA, SciDocs, Climate-FEVER, HotpotQA, SciFact, NFCorpus, DBPedia-entity, FEVER. Holm áp dụng trên cả 8 bộ trong mỗi họ nên chặt hơn Step 10-B2. Bộ ablation của HotpotQA và FEVER là 1.500 query đầu.

Thêm một bảng mô tả (không kiểm định): trọng số LLM trung bình mà mỗi danh sách OS, OD, ES, ED nhận được ở từng chế độ, để xem thứ tự cố định (ES, OS, ED, OD) đang ưu ái danh sách nào.

- **Accelerator:** None (CPU). **Internet:** On 
- **Input:** `qudar-runs`, `qudar-runs-2`, `qudar-beir-data`, `qudar-beir-data-2`, `qudar-llm-qwen`. Không cần `qudar-patches`.
- **Thời gian ước tính:** cài đặt 1–2 phút; đọc `retrieval_result.json` của HotpotQA và FEVER mỗi bộ 3–5 phút; tính chỉ số từng query và kiểm định 15–30 phút. **Tổng khoảng 30–60 phút.**

In [1]:
import glob, os
def first(pattern, what, keep=lambda p: True):
    hits = sorted(p for p in glob.glob(pattern, recursive=True) if keep(p))
    assert hits, f"không tìm thấy {what} ({pattern}): kiểm tra dataset đã gắn chưa"
    return hits[0]

!git clone -q --depth 1 https://github.com/KarryChuong/CS419-IR-QuDAR-replication.git /kaggle/working/repo
REPO = "/kaggle/working/repo"
assert "weights_by_source" in open(f"{REPO}/scripts/judge_significance.py").read(), "main chưa có judge_significance.py bản mới"
assert os.path.exists(f"{REPO}/results/step10/llm_qwen/llm_scores_paper_fever_test.csv"), "main chưa có điểm Qwen của 4 bộ mới"

# Gộp run và dữ liệu cũ + mới vào một chỗ bằng symlink (script chỉ nhận một --runs và một --data-root)
OLD  = ["fiqa_test", "scidocs_test", "climate-fever_test", "hotpotqa_test"]
NEW  = ["scifact_test", "nfcorpus_test", "dbpedia-entity_test", "fever_test"]
RUNS_OLD = os.path.dirname(os.path.dirname(first("/kaggle/input/**/fiqa_test/retrieval_result.json", "qudar-runs")))
RUNS_NEW = os.path.dirname(os.path.dirname(first("/kaggle/input/**/fever_test/retrieval_result.json", "qudar-runs-2")))
DATA_OLD = os.path.dirname(os.path.dirname(first("/kaggle/input/**/qudar-beir-data/**/fiqa/corpus.jsonl", "qudar-beir-data")))
DATA_NEW = os.path.dirname(os.path.dirname(first("/kaggle/input/**/qudar-beir-data-2/**/dbpedia-entity/corpus.jsonl", "qudar-beir-data-2")))
RUNS, DATA = "/kaggle/working/runs_all", "/kaggle/working/data_all"
os.makedirs(RUNS, exist_ok=True); os.makedirs(DATA, exist_ok=True)
for names, src in [(OLD, RUNS_OLD), (NEW, RUNS_NEW)]:
    for n in names:
        assert os.path.exists(f"{src}/{n}/retrieval_result.json"), f"thiếu run {n} trong {src}"
        if not os.path.exists(f"{RUNS}/{n}"):
            os.symlink(f"{src}/{n}", f"{RUNS}/{n}")
for names, src in [(OLD, DATA_OLD), (NEW, DATA_NEW)]:
    for n in names:
        ds = n.rsplit("_", 1)[0]
        assert os.path.exists(f"{src}/{ds}/qrels/test.tsv"), f"thiếu qrels {ds} trong {src}"
        if not os.path.exists(f"{DATA}/{ds}"):
            os.symlink(f"{src}/{ds}", f"{DATA}/{ds}")

QWEN_OLD = os.path.dirname(first("/kaggle/input/**/llm_scores_paper_fiqa_test.csv", "qudar-llm-qwen",
                                 keep=lambda p: "qwen" in p.lower()))
GPT  = ",".join(f"{REPO}/results/{d}" for d in ["step4_llm/test", "step6_ablation", "step10/llm_gpt"])
QWEN = f"{QWEN_OLD},{REPO}/results/step10/llm_qwen"
FULL = OLD + NEW
ABL  = ["fiqa_test", "scidocs_test", "climate-fever_test", "hotpotqa_test_first1500",
        "scifact_test", "nfcorpus_test", "dbpedia-entity_test", "fever_test_first1500"]
FULL_S, ABL_S = " ".join(FULL), " ".join(ABL)
!pip install -q tabulate
print("RUNS_OLD =", RUNS_OLD, "\nRUNS_NEW =", RUNS_NEW, "\nDATA_OLD =", DATA_OLD, "\nDATA_NEW =", DATA_NEW,
      "\nGPT  =", GPT, "\nQWEN =", QWEN)

RUNS_OLD = /kaggle/input/datasets/karrychuong/qudar-runs/runs 
RUNS_NEW = /kaggle/input/datasets/karrychuong/qudar-runs-2/runs 
DATA_OLD = /kaggle/input/datasets/karrychuong/qudar-beir-data/data 
DATA_NEW = /kaggle/input/datasets/karrychuong/qudar-beir-data-2/data 
GPT  = /kaggle/working/repo/results/step4_llm/test,/kaggle/working/repo/results/step6_ablation,/kaggle/working/repo/results/step10/llm_gpt 
QWEN = /kaggle/input/datasets/karrychuong/qudar-llm-qwen/llm_qwen,/kaggle/working/repo/results/step10/llm_qwen


In [2]:
!python {REPO}/scripts/judge_significance.py --runs {RUNS} --data-root {DATA} \
    --judge gpt-4o-mini={GPT} --judge Qwen2.5-7B={QWEN} --out /kaggle/working/step10d1 \
    --full {FULL_S} --ablation {ABL_S}

[06:55:54] fiqa_test: loading retrieval_result.json
[06:56:01] fiqa_test: 648 queries, 8 LLM runs
[06:56:01] scidocs_test: loading retrieval_result.json
[06:56:12] scidocs_test: 1000 queries, 8 LLM runs
[06:56:12] climate-fever_test: loading retrieval_result.json
[06:56:32] climate-fever_test: 1535 queries, 8 LLM runs
[06:56:32] hotpotqa_test: loading retrieval_result.json
[06:57:22]   hotpotqa_test: 2000/7405 queries
[06:57:26]   hotpotqa_test: 4000/7405 queries
[06:57:31]   hotpotqa_test: 6000/7405 queries
[06:57:34] hotpotqa_test: 7405 queries, 2 LLM runs
[06:57:40] hotpotqa_test_first1500: 1500 queries, 8 LLM runs
[06:57:41] scifact_test: loading retrieval_result.json
[06:57:45] scifact_test: 300 queries, 8 LLM runs
[06:57:45] nfcorpus_test: loading retrieval_result.json
[06:57:48] nfcorpus_test: 323 queries, 8 LLM runs
[06:57:48] dbpedia-entity_test: loading retrieval_result.json
[06:57:54] dbpedia-entity_test: 400 queries, 8 LLM runs
[06:57:54] fever_test: loading retrieval_resul

In [3]:
# Tóm tắt
import json, shutil
import pandas as pd
pd.set_option("display.width", 300); pd.set_option("display.max_colwidth", 50)
OUT = "/kaggle/working/step10d1"
summary = json.load(open(f"{OUT}/summary.json"))
if summary["missing"]:
    print("!!! THIẾU FILE ĐIỂM, kết quả chưa đủ:", summary["missing"])

# 1. Chênh lệch A - B theo từng bộ, dấu * = có ý nghĩa sau Holm (Holm trên cả 8 bộ trong mỗi họ)
t = pd.read_csv(f"{OUT}/sig_tests.csv")
t["cell"] = t["mean(A-B)"].map("{:+.4f}".format) + t["sig (Holm p<0.05)"].fillna("")
cols = list(dict.fromkeys(t["set"]))
pv = t.pivot_table(index=["family", "judge", "comparison", "metric"], columns="set", values="cell",
                   aggfunc="first", sort=False)[cols]
print(pv.fillna("").to_string())

# 2. Thiên lệch vị trí
pos = pd.read_csv(f"{OUT}/position_tests.csv")
print("\n" + pos[["judge", "set", "mode", "queries", "pos 1", "pos 2", "pos 3", "pos 4", "highest", "lowest",
                  "p Holm", "sig (Holm p<0.05)"]].fillna("").to_string(index=False))

# 3. Trọng số trung bình: thứ tự cố định - thứ tự xáo, theo từng danh sách
w = pd.read_csv(f"{OUT}/weights_by_source.csv")
W = [f"weight {x}" for x in ["OS", "OD", "ES", "ED"]]
rows = []
for (j, s), g in w.groupby(["judge", "set"], sort=False):
    g = g.set_index("mode")
    for prompt in ["paper", "code"]:
        fx, sh = f"{prompt} prompt + fixed", f"{prompt} prompt + shuffled"
        if fx in g.index and sh in g.index:
            d = g.loc[fx, W] - g.loc[sh, W]
            rows.append({"judge": j, "set": s, "prompt": prompt, **{c.replace("weight", "Δ"): round(float(v), 3) for c, v in d.items()}})
print("\nTrọng số LLM trung bình: cố định - xáo (thứ tự cố định = ES, OS, ED, OD)")
print(pd.DataFrame(rows).to_string(index=False))

# 4. Đối chiếu với số đã báo cáo
chk = pd.read_csv(f"{OUT}/check_means.csv")
print("\nĐối chiếu với số đã báo cáo: max |diff| =", chk["max |diff|"].max())
print(chk[chk["max |diff|"] > 0.001][["set", "method", "ours Recall@10", "reported", "max |diff|"]].to_string(index=False))
print(json.dumps(summary["families"], indent=1, ensure_ascii=False))

dst = "/kaggle/working/for_repo/results/step10/sig8"
shutil.copytree(OUT, dst, dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/for_repo_step10d1", "zip", "/kaggle/working/for_repo")
!du -sh {OUT} /kaggle/working/for_repo_step10d1.zip

set                                                                                                                   FiQA  SciDocs Climate-FEVER  HotpotQA  SciFact  NFCorpus DBPedia-entity     FEVER HotpotQA (first 1500) FEVER (first 1500)
family          judge                    comparison                                                    metric                                                                                                                                   
F1 llm_vs_equal gpt-4o-mini              QuDAR-llm (paper prompt + shuffled) - Equal                   Recall@10   -0.0053  -0.0002       +0.0063  -0.0123*  -0.0007   +0.0030        -0.0129   -0.0030                                         
                                                                                                       nDCG@10     +0.0027  +0.0006       +0.0046   +0.0037  -0.0005   +0.0020        -0.0016  +0.0164*                                         
                                    